# 01 Inference Baseline

This notebook runs inference using trained YOLOv8 weights on validation and new images.


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: Tesla T4


In [3]:
!pip -q install ultralytics==8.* roboflow

import sys, torch, platform
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
CURVES_DIR = RESULTS_DIR / "curves"
EVIDENCE_DIR = RESULTS_DIR / "evidence"
for p in [RESULTS_DIR, CURVES_DIR, EVIDENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

!pip show ultralytics | sed -n '1,5p'
print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 82.4 MB/s eta 0:00:00
Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cpu
CUDA: False
Name: ultralytics
Version: 8.4.19
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
ROOT: /content
RESULTS_DIR: /content/results


In [25]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="w36YxNlsftoXK1pSr57D")
project = rf.workspace("zigurat-assignment").project("ppe-hardhat-detection")
version = project.version(2)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [28]:
from roboflow import Roboflow
from pathlib import Path

RF_API_KEY = "w36YxNlsftoXK1pSr57D"
WORKSPACE = "zigurat-assignment"
PROJECT = "ppe-hardhat-detection"
VERSION = 2  # changed to your dataset version

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset dir:", DATASET_DIR)
print("Data yaml exists:", DATA_YAML.exists())

loading Roboflow workspace...
loading Roboflow project...
Dataset dir: /content/PPE-HardHat-Detection-2
Data yaml exists: True


In [29]:
from ultralytics import YOLO

MODEL_VARIANT = "yolov8n.pt"  # change to yolov8s.pt if you trained s
VERIFY_EPOCHS = 5
BATCH = 16
IMGSZ = 640

model = YOLO(MODEL_VARIANT)
model.train(
    data=str(DATA_YAML),
    epochs=VERIFY_EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=str(RESULTS_DIR),
    name="verify_train",
    exist_ok=True
)

print("Verification training complete.")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PPE-HardHat-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=verify_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [4]:
from ultralytics import YOLO
from pathlib import Path

# --- Training configuration (document these in README) ---
MODEL_VARIANT = "yolov8n.pt"   # change to yolov8s.pt if you trained s
EPOCHS = 30                    # >=30 required
BATCH = 16                     # adjust if you get out-of-memory
IMGSZ = 640                    # common YOLO default

RESULTS_DIR = Path("results")
RUN_NAME = "train_30ep"

model = YOLO(MODEL_VARIANT)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RESULTS_DIR),
    name=RUN_NAME,
    exist_ok=True
)

print("Training complete.")
print("Run directory:", RESULTS_DIR / RUN_NAME)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


NameError: name 'DATA_YAML' is not defined

In [ ]:
from pathlib import Path

RUN_DIR = Path("results") / RUN_NAME
WEIGHTS_PATH = RUN_DIR / "weights" / "best.pt"

print("Best weights path:", WEIGHTS_PATH)
print("Weights exist:", WEIGHTS_PATH.exists())

In [ ]:
from pathlib import Path

RUN_DIR = Path("results") / RUN_NAME
WEIGHTS_PATH = RUN_DIR / "weights" / "best.pt"

print("Best weights path:", WEIGHTS_PATH)
print("Weights exist:", WEIGHTS_PATH.exists())

In [ ]:
import shutil
from pathlib import Path

CURVES_DIR = Path("results") / "curves"
CURVES_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR = Path("results") / RUN_NAME

# Common Ultralytics plot filenames (some may not exist depending on version)
PLOTS = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

copied = []
for fname in PLOTS:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, CURVES_DIR / fname)
        copied.append(fname)

print("Copied plots:", copied)
print("Curves saved to:", CURVES_DIR)

In [ ]:
from pathlib import Path

VAL_IMG_DIR = DATASET_DIR / "valid" / "images"
val_imgs = sorted(list(VAL_IMG_DIR.glob("*")))

print("Validation images found:", len(val_imgs))
sample_val = val_imgs[:10]

best_model.predict(
    source=[str(p) for p in sample_val],
    save=True,
    project="results",
    name="val_preds",
    exist_ok=True
)

print("Saved validation predictions to:", Path("results") / "val_preds")

In [ ]:
from pathlib import Path

NEW_IMG_DIR = Path("new_images")
NEW_IMG_DIR.mkdir(exist_ok=True)

new_imgs = sorted(list(NEW_IMG_DIR.glob("*")))
print("New images found:", len(new_imgs))

sample_new = new_imgs[:5]

best_model.predict(
    source=[str(p) for p in sample_new],
    save=True,
    project="results",
    name="new_preds",
    exist_ok=True
)

print("Saved new image predictions to:", Path("results") / "new_preds")

In [ ]:
import shutil
from pathlib import Path

EVIDENCE_BASE = Path("results") / "evidence"
VAL_EVID = EVIDENCE_BASE / "val_preds"
NEW_EVID = EVIDENCE_BASE / "new_preds"

VAL_EVID.mkdir(parents=True, exist_ok=True)
NEW_EVID.mkdir(parents=True, exist_ok=True)

# YOLO saves predicted images in: results/<name>/
VAL_PRED_DIR = Path("results") / "val_preds"
NEW_PRED_DIR = Path("results") / "new_preds"

# Copy predicted JPG/PNG outputs to evidence folders
def copy_preds(src_dir: Path, dst_dir: Path, limit: int):
    imgs = sorted([p for p in src_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    for p in imgs[:limit]:
        shutil.copy(p, dst_dir / p.name)
    return [p.name for p in imgs[:limit]]

val_copied = copy_preds(VAL_PRED_DIR, VAL_EVID, 10)
new_copied = copy_preds(NEW_PRED_DIR, NEW_EVID, 5)

print("Copied val evidence:", val_copied)
print("Copied new evidence:", new_copied)
print("Evidence base:", EVIDENCE_BASE)

In [ ]:
import time, platform, torch
from ultralytics import __version__ as ulty_ver

print("Reproducibility Proof")
print("- Date/Time (UTC):", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
print("- Platform:", platform.platform())
print("- Ultralytics:", ulty_ver)
print("- CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("- GPU:", torch.cuda.get_device_name(0))
print(f"- Train config: model={MODEL_VARIANT}, epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}")
print("- Run directory:", str(Path("results") / RUN_NAME))
print("- Best weights:", str(WEIGHTS_PATH))